In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
col_names = [
    "white_king_file", "white_king_rank",
    "white_rook_file", "white_rook_rank",
    "black_king_file", "black_king_rank",
    "result"
]
os.chdir(r"C:\Users\klemi\Documents\studia\sem3\twd")
df = pd.read_csv("krkopt.data", names=col_names)

In [ ]:
mapping = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5, 'f': 6, 'g': 7, 'h': 8}
for col in ["white_king_file", "white_rook_file", "black_king_file"]:
    df[col.replace('file', 'file_num')] = df[col].map(mapping)

def parse_result(x):
    if x == 'draw': return -1
    if x == 'zero': return 0
    if x == 'one': return 1
    if x == 'two': return 2
    if x == 'three': return 3
    if x == 'four': return 4
    if x == 'five': return 5
    if x == 'six': return 6
    if x == 'seven': return 7
    if x == 'eight': return 8
    if x == 'nine': return 9
    if x == 'ten': return 10
    if x == 'eleven': return 11
    if x == 'twelve': return 12
    if x == 'thirteen': return 13
    if x == 'fourteen': return 14
    if x == 'fifteen': return 15
    if x == 'sixteen': return 16
    return int(x)

df['depth'] = df['result'].apply(parse_result)

#### Pytanie 2

In [ ]:

dx = abs(df['white_king_file_num'] - df['black_king_file_num'])
dy = abs(df['white_king_rank'] - df['black_king_rank'])

df['dist_euclidean'] = np.sqrt(dx**2 + dy**2)
df['dist_manhattan'] = dx + dy
df['dist_chebyshev'] = np.maximum(dx, dy)

#### Pytanie 3

In [ ]:
df['bk_dist_edge_x'] = df['black_king_file_num'].apply(lambda x: min(x-1, 8-x))
df['bk_dist_edge_y'] = df['black_king_rank'].apply(lambda x: min(x-1, 8-x))
df['bk_dist_edge'] = df[['bk_dist_edge_x', 'bk_dist_edge_y']].min(axis=1)

In [ ]:
df.head(15)

### Wizualizacje

#### Pytanie 1: Gdzie jest kr?l przy macie/remisie?

In [ ]:
fig1, axes = plt.subplots(1, 2, figsize=(12, 5))

# Mat
mate_counts = df[df['depth'] == 0].groupby(['black_king_rank', 'black_king_file']).size().unstack(fill_value=0)
mate_counts = mate_counts.reindex(index=range(1, 9), columns=['a','b','c','d','e','f','g','h'], fill_value=0).sort_index(ascending=False)
sns.heatmap(mate_counts, ax=axes[0], cmap='Reds', cbar=False)
axes[0].set_title('Pozycja Czarnego Kr?la (Mat)')

# Remis
draw_counts = df[df['depth'] == -1].groupby(['black_king_rank', 'black_king_file']).size().unstack(fill_value=0)
draw_counts = draw_counts.reindex(index=range(1, 9), columns=['a','b','c','d','e','f','g','h'], fill_value=0).sort_index(ascending=False)
sns.heatmap(draw_counts, ax=axes[1], cmap='Blues', cbar=False)
axes[1].set_title('Pozycja Czarnego Kr?la (Remis)')
plt.show()

##### Lewa heatmap'a pokazuje, ?e gdy czarny kr?l jest matowany najcz??ciej znajduje si? on w lewym rogu planszy. Rzadko przy takim usytuowaniu zachodzi remis, wr?cz prawie wcale co obrazuje prawa heatmapa. Remisem gra ko?czy si? najcz?sciej przy pozycji kr?la na polach od 2g do 6g.

#### Pytanie 2: Metryki vs G??boko?? (bez remis?w)

In [ ]:

df_2 = df[df['depth'] != -1]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.boxplot(x='depth', y='dist_euclidean', data=df_2, ax=axes[0], color='lightblue')
axes[0].set_title('Metryka Euklidesowa')
axes[0].set_xlabel('Liczba ruch?w do ko?ca')
axes[0].set_ylabel('Dystans pomi?dzy bia?ym kr?lem a czarnym kr?lem')


sns.boxplot(x='depth', y='dist_manhattan', data=df_2, ax=axes[1], color='lightgreen')
axes[1].set_title('Metryka Manhattan')
axes[1].set_xlabel('Liczba ruch?w do ko?ca')
axes[1].set_ylabel('Dystans pomi?dzy bia?ym kr?lem a czarnym kr?lem')


sns.boxplot(x='depth', y='dist_chebyshev', data=df_2, ax=axes[2], color='lightpink')
axes[2].set_title('Metryka Czebyszewa')
axes[2].set_xlabel('Liczba ruch?w do ko?ca')
axes[2].set_ylabel('Dystans pomi?dzy bia?ym kr?lem a czarnym kr?lem')

plt.tight_layout()
plt.show()

#### Pytanie 3: Kraw?d? vs G??boko??

In [ ]:
df_3 = df[df['depth'] != -1].copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='bk_dist_edge', y='depth', data=df_3, ax=axes[0], palette="viridis")

axes[0].set_title('Wp?yw odleg?o?ci czarnego kr?la od kraw?dzi na liczb? ruch?w do wygranej bia?ych')
axes[0].set_xlabel('Liczba ruch?w potrzebnych by czarny kr?l znalaz? si? przy kraw?dzi')
axes[0].set_ylabel('Liczba ruch?w do wygranej bia?ych')

heatmap_data = df_3.groupby(['bk_dist_edge', 'dist_chebyshev'])['depth'].mean().unstack()
sns.heatmap(heatmap_data, ax=axes[1], cmap="coolwarm", annot=True, fmt=".1f")

axes[1].set_title('?rednia liczba ruch?w do wygranej bia?ych')
axes[1].set_ylabel('Liczba ruch?w potrzebnych by czarny kr?l znalaz? si? przy kraw?dzi')
axes[1].set_xlabel('Dystans mi?dzy Kr?lami (Czebyszew)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

##### Lewy wykres:
##### Na wykresie mo?na zauwa?y? pewien trend: w miar? jak czarny kr?l przesuwa si? do centrum i oddala od kraw?dzi, ?rednia liczba ruch?w do wygranej dla bia?ych pionkow zauwa?alnie wzrasta. 
##### Prawy wykres:
##### Lewy dolny r?g wykresu ukazuje sytuacje, gdy czarny kr?l jest na kraw?dzi, a bia?y kr?l jest blisko niego. Dzi?ki takiemu uk?adowi ?rednia liczba ruch?w do wygranej bia?ych jest najni?sza. Za? prawa g?rna cz??? wykresu obrazuje najtrudniejsz? sytuacj? dla bia?ych, czyli czarny kr?l jest w centrum, a bia?y kr?l jest daleko. Wtedy te? ?rednia liczba ruch?w do wygranej bia?ych jest najwy?sza.